# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [1]:
import numpy as np

# Datos iniciales del enunciado
query = "latency vs throughput"
docs = [
    "doc0: Latency refers to the time it takes for a single data packet to travel.",
    "doc1: Throughput is the total amount of data successfully processed over time.",
    "doc2: Network configuration guide for local routers and switches.",
    "doc3: High latency reduces throughput performance in TCP connections.",
    "doc4: Financial report on cloud infrastructure spending quarterly."
]

# Scores simulados alineados con los 5 documentos
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3])

print("--- ETAPA 1: FILTRADO CON BI-ENCODER ---")
# 1. Obtener los índices ordenados por Bi-Encoder de mayor a menor score
# [::-1] invierte el resultado de argsort para que vaya de mayor a menor
indices_bi_ordenados = np.argsort(bi_scores)[::-1]

# 2. Quedarnos estrictamente con el Top-4
top_4_indices = indices_bi_ordenados[:4]
print(f"Índices seleccionados por Bi-Encoder (Top-4): {top_4_indices}")
print(f"Documentos descartados en esta etapa: {indices_bi_ordenados[4:]}\n")


print("--- ETAPA 2: REORDENAMIENTO CON CROSS-ENCODER ---")
# 3. Extraer los cross_scores únicamente de esos 4 documentos seleccionados
scores_candidatos = cross_scores[top_4_indices]

# 4. Ordenar esos candidatos según su Cross-Score (de mayor a menor)
# np.argsort(scores_candidatos)[::-1] nos da el orden relativo dentro del Top-4
indices_reordenados_relativos = np.argsort(scores_candidatos)[::-1]

# 5. Mapear de vuelta a los índices originales del dataset general
indices_finales = top_4_indices[indices_reordenados_relativos]


# Visualizar el orden definitivo en el laboratorio
print("ORDEN FINAL DEL RANKING REORDENADO:")
print("----------------------------------------------------------------")
for posicion, idx in enumerate(indices_finales, start=1):
    print(f"Puesto #{posicion} | {docs[idx]} | Bi-Score: {bi_scores[idx]:.2f} | Cross-Score: {cross_scores[idx]:.2f}")

--- ETAPA 1: FILTRADO CON BI-ENCODER ---
Índices seleccionados por Bi-Encoder (Top-4): [1 3 0 4]
Documentos descartados en esta etapa: [2]

--- ETAPA 2: REORDENAMIENTO CON CROSS-ENCODER ---
ORDEN FINAL DEL RANKING REORDENADO:
----------------------------------------------------------------
Puesto #1 | doc1: Throughput is the total amount of data successfully processed over time. | Bi-Score: 0.81 | Cross-Score: 0.90
Puesto #2 | doc3: High latency reduces throughput performance in TCP connections. | Bi-Score: 0.78 | Cross-Score: 0.85
Puesto #3 | doc4: Financial report on cloud infrastructure spending quarterly. | Bi-Score: 0.60 | Cross-Score: 0.30
Puesto #4 | doc0: Latency refers to the time it takes for a single data packet to travel. | Bi-Score: 0.72 | Cross-Score: 0.10


## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [2]:
def mmr_selection(query_vec, candidate_vecs, lambda_param=0.5, k=3):
    selected_indices = []
    remaining_indices = list(range(len(candidate_vecs)))

    # Bucle principal para rellenar nuestro Top-K de forma diversa
    while len(selected_indices) < k and remaining_indices:
        # 1. Calcular la similitud de cada candidato restante con la query
        sim_to_query = [cosine_sim(candidate_vecs[i], query_vec) for i in remaining_indices]

        # 2. Calcular la máxima similitud de cada candidato con los que ya han sido seleccionados
        max_sim_to_selected = [max([cosine_sim(candidate_vecs[i], candidate_vecs[j]) for j in selected_indices]) if selected_indices else 0 for i in remaining_indices]

        # 3. Aplicar la ecuación MMR: balancear relevancia (lambda) y redundancia (1 - lambda)
        mmr_scores = [lambda_param * sq - (1 - lambda_param) * ms for sq, ms in zip(sim_to_query, max_sim_to_selected)]

        # 4. Encontrar el índice del candidato con el score MMR más alto
        best_candidate_idx = remaining_indices[np.argmax(mmr_scores)]

        # 5. Mover el elegido de la lista de pendientes a la lista de seleccionados
        selected_indices.append(best_candidate_idx)
        remaining_indices.remove(best_candidate_idx)

    return selected_indices


## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa | Coste relativo (tú eliges

---

escala) |
|--------|-------------------------------------|
| ... | ... |

Estimación de costes y latencia en producciónPara entender el cuello de botella del pipeline, he armado esta tabla comparativa con el impacto estimado de cada operación en tiempo (latencia) y coste computacional:
Operación,Latencia aproximada,Coste computacional / Económico,Rol en el pipeline
Embedding único de la Query,Menos de 10 - 20 ms,Prácticamente gratis (Modelos muy ligeros como all-MiniLM),Ultra rápido: Es el paso inicial para poder buscar en FAISS.
K llamadas al Cross-Encoder,50 - 150 ms (Depende del valor de K),"Medio (Sigue siendo local, pero requiere procesar texto cruzado)","Moderado: Es más pesado porque compara la query con cada chunk a la vez, pero al aplicarlo solo a un K pequeño (ej. K=4), es totalmente viable."
Generación LLM (200 tokens),1000 - 3000 ms (1 a 3 segundos),Muy alto (Pago por tokens en API o uso intensivo de GPU si es local),"El cuello de botella: Generar texto palabra por palabra mediante autoregresión es, con diferencia, lo más lento y caro de todo el proceso."
